In [20]:
# Installing all required packages
!pip install openai scikit-learn pandas numpy

Defaulting to user installation because normal site-packages is not writeable


In [21]:
!pip install openai

Defaulting to user installation because normal site-packages is not writeable


In [22]:
import sys
!{sys.executable} -m pip install openai

Defaulting to user installation because normal site-packages is not writeable


In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from openai import AzureOpenAI
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully")

✅ All libraries imported successfully


In [6]:
# Load both CSV files
meta_df = pd.read_csv(r"C:\Users\Priya\Downloads\Project_JOB_meta_data.csv")
schedule_df = pd.read_csv(r"C:\Users\Priya\Downloads\Project_Schedule_Data.csv")

print(f"Meta data shape: {meta_df.shape}")       # should show (564, 11)
print(f"Schedule data shape: {schedule_df.shape}") # should show (~91000, 5)

# Quick preview
display(meta_df.head(3))
display(schedule_df.head(3))

Meta data shape: (1129, 11)
Schedule data shape: (91106, 5)


,CONTROL_JOB_NO,REGION_NAME,BUSINESS_UNIT,OFFICE_NAME,CITY,STATE,CORE_MARKET,MARKET_SECTOR,PROJECT_TYPE,TOTAL_CONTROL_ESTIMATE,BUILDING_SIZE
0,PROJECT_0524,Northeast,Boston,Boston,Worcester,MA,Clinical Infrastructure,CI-Healthcare Provider,Hospitals,1344709.0,1520.0
1,PROJECT_0061,Northwest,Sacramento,Sacramento,Santa Rosa,CA,Clinical Infrastructure,NaN,Hospitals,0.0,0.0
2,PROJECT_0139,Southwest,SoCal,Newport Beach,Los Angeles,CA,Clinical Infrastructure,NaN,Hospitals,777668.0,NaN


,MAIN_PROJECTID,PARWBSNAME,WBSNAME,PLANNEDSTARTDATE,PLANNEDFINISHDATE
0,PROJECT_0001,ARE 600 Gateway Myoventive TI Projects,Contract Milestones,2026-01-16,2026-01-20
1,PROJECT_0001,ARE 600 Gateway Myoventive TI Projects,Project Milestones,2025-11-03,2026-01-16
2,PROJECT_0001,Construction,Interior Construction,2025-12-19,2026-01-14


In [7]:
print("=== META DATA ===")
print("\nColumn-wise null counts:")
print(meta_df.isnull().sum())

print("\nNumerical columns stats:")
display(meta_df[['TOTAL_CONTROL_ESTIMATE', 'BUILDING_SIZE']].describe())

print("\nProject type distribution:")
print(meta_df['PROJECT_TYPE'].value_counts().head(10))

print("\n=== SCHEDULE DATA ===")
print("Unique project IDs:", schedule_df['MAIN_PROJECTID'].nunique())
print("\nTop WBS task names:")
print(schedule_df['WBSNAME'].value_counts().head(10))

=== META DATA ===

Column-wise null counts:
CONTROL_JOB_NO              0
REGION_NAME                56
BUSINESS_UNIT              56
OFFICE_NAME                28
CITY                      113
STATE                     106
CORE_MARKET                 7
MARKET_SECTOR             546
PROJECT_TYPE                9
TOTAL_CONTROL_ESTIMATE      0
BUILDING_SIZE             102
dtype: int64

Numerical columns stats:


,TOTAL_CONTROL_ESTIMATE,BUILDING_SIZE
count,1.129000e+03,1.027000e+03
mean,2.450643e+07,1.323483e+05
std,1.103020e+08,4.234478e+05
min,-2.608607e+07,0.000000e+00
25%,6.025700e+04,4.500000e+03
50%,9.410460e+05,2.100000e+04
75%,5.753108e+06,1.250000e+05
max,1.407318e+09,7.500000e+06



Project type distribution:
PROJECT_TYPE
Hospitals                          267
Mission Critical                   231
Office                             143
R&amp;D Building (Laboratories)    121
Biopharm Manufacturing              87
Outpatient Healthcare               78
Other                               61
Education/Classroom Space           58
Advanced Manufacturing              28
Sports and Athletics                13
Name: count, dtype: int64

=== SCHEDULE DATA ===
Unique project IDs: 572

Top WBS task names:
WBSNAME
Infrastructure                 397
Design                         327
Project Milestones             302
Finishes                       288
Cabling                        273
Network                        269
BID PERIOD                     224
META REVIEW / AUTHORIZATION    224
Permits                        210
TRADE PARTNER AWARDS           208
Name: count, dtype: int64


In [8]:
meta_clean = meta_df.copy()

# Filling missing numerical values with median
meta_clean['TOTAL_CONTROL_ESTIMATE'] = meta_clean['TOTAL_CONTROL_ESTIMATE'].fillna(
    meta_clean['TOTAL_CONTROL_ESTIMATE'].median()
)
meta_clean['BUILDING_SIZE'] = meta_clean['BUILDING_SIZE'].fillna(
    meta_clean['BUILDING_SIZE'].median()
)

# Filling missing text columns with 'Unknown'
text_cols = ['REGION_NAME', 'BUSINESS_UNIT', 'OFFICE_NAME', 'CORE_MARKET', 
             'MARKET_SECTOR', 'PROJECT_TYPE', 'STATE', 'CITY']
for col in text_cols:
    meta_clean[col] = meta_clean[col].fillna('Unknown')

# Encode text columns into numbers using LabelEncoder
# Machine learning can only work with numbers
label_encoders = {}
categorical_cols = ['REGION_NAME', 'BUSINESS_UNIT', 'OFFICE_NAME', 
                    'CORE_MARKET', 'MARKET_SECTOR', 'PROJECT_TYPE', 'STATE', 'CITY']

for col in categorical_cols:
    le = LabelEncoder()
    meta_clean[col + '_ENC'] = le.fit_transform(meta_clean[col])
    label_encoders[col] = le  # save encoder for later use on new input

print("✅ Cleaning done. New encoded columns added.")
print(meta_clean[[c + '_ENC' for c in categorical_cols]].head(3))

✅ Cleaning done. New encoded columns added.
   REGION_NAME_ENC  BUSINESS_UNIT_ENC  OFFICE_NAME_ENC  CORE_MARKET_ENC  \
0                1                  4                3                3   
1                2                 15               20                3   
2                4                 17               14                3   

   MARKET_SECTOR_ENC  PROJECT_TYPE_ENC  STATE_ENC  CITY_ENC  
0                 15                 3          9       224  
1                 31                 3          2       179  
2                 31                 3          2       113  


In [9]:
# These are the feature columns we'll use to measure similarity
feature_cols = [
    'REGION_NAME_ENC', 'BUSINESS_UNIT_ENC', 'CORE_MARKET_ENC',
    'MARKET_SECTOR_ENC', 'PROJECT_TYPE_ENC',
    'TOTAL_CONTROL_ESTIMATE', 'BUILDING_SIZE'
]

# Extract feature matrix
X = meta_clean[feature_cols].values
# Scale features so all are on the same scale (0 to 1 range)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Feature matrix shape: {X_scaled.shape}")
print("✅ Features scaled successfully")

Feature matrix shape: (1129, 7)
✅ Features scaled successfully


In [12]:
# These are the EXACT categorical columns used in feature_cols for scaling
# (OFFICE_NAME and STATE are excluded — they were encoded but not used in the model)
model_categorical_cols = ['REGION_NAME', 'BUSINESS_UNIT', 'CORE_MARKET', 
                          'MARKET_SECTOR', 'PROJECT_TYPE']

# Compute cosine similarity matrix (this part stays the same)
similarity_matrix = cosine_similarity(X_scaled)

def find_similar_projects(new_project_dict, top_n=3):
    """
    Given a new project's details, find the top N most similar historical projects.
    """
    
    new_features = []
    
    # Step 1: Only loop through model_categorical_cols (5 columns)
    # NOT the full categorical_cols (7 columns)
    for col in model_categorical_cols:
        val = new_project_dict.get(col, 'Unknown')
        le = label_encoders[col]
        
        # Handle unseen categories gracefully
        if val in le.classes_:
            encoded = le.transform([val])[0]
        else:
            encoded = 0  # default for unknown value
        
        new_features.append(encoded)
    
    # Step 2: Append the 2 numerical features
    new_features.append(new_project_dict.get('TOTAL_CONTROL_ESTIMATE', 0))
    new_features.append(new_project_dict.get('BUILDING_SIZE', 0))
    
    # Confirm feature count before scaling
    print(f"   Feature count being passed to scaler: {len(new_features)}")
    print(f"   Scaler expects: {scaler.n_features_in_} features")
    
    # Step 3: Scale using the same scaler
    new_vector = scaler.transform([new_features])
    
    # Step 4: Compute cosine similarity
    scores = cosine_similarity(new_vector, X_scaled)[0]
    
    # Step 5: Get top N similar projects
    top_indices = np.argsort(scores)[::-1][:top_n]
    similar_project_ids = meta_clean.iloc[top_indices]['CONTROL_JOB_NO'].tolist()
    similarity_scores = scores[top_indices].tolist()
    
    print(f"\n🔍 Top {top_n} similar projects found:")
    for pid, score in zip(similar_project_ids, similarity_scores):
        print(f"  → {pid} | Similarity Score: {score:.4f}")
    
    return similar_project_ids

print("✅ Similarity function ready")

✅ Similarity function ready


In [13]:
def retrieve_schedules(project_ids):
    """
    Given a list of project IDs, retrieve their schedule (WBS tasks) from schedule_df.
    
    Returns a formatted string ready to pass to the LLM.
    """
    
    schedules_text = ""
    
    for pid in project_ids:
        project_schedule = schedule_df[schedule_df['MAIN_PROJECTID'] == pid]
        
        if project_schedule.empty:
            schedules_text += f"\n[No schedule found for {pid}]\n"
            continue
        
        schedules_text += f"\n--- Schedule for Project {pid} ---\n"
        
        for _, row in project_schedule.iterrows():
            task_line = (
                f"  Parent: {row['PARWBSNAME']} | "
                f"Task: {row['WBSNAME']} | "
                f"Start: {row['PLANNEDSTARTDATE']} | "
                f"End: {row['PLANNEDFINISHDATE']}\n"
            )
            schedules_text += task_line
    
    print(f"✅ Retrieved schedules for {len(project_ids)} projects")
    print(f"   Total characters in schedule text: {len(schedules_text)}")
    
    return schedules_text

print("✅ Schedule retrieval function ready")

✅ Schedule retrieval function ready


In [14]:
# Configure Azure OpenAI client
client = AzureOpenAI(
    azure_endpoint = "https://vcon.openai.azure.com/",
    api_key = "EfwhbUa3MrbLrioEAyALwcjwffIAOQzr6rtdeRXbTvgROOyrl9pmJQQJ99CAACYeBjFXJ3w3AAABACOGUCez",
    api_version = "2024-12-01-preview"
)

DEPLOYMENT_NAME = "gpt-4o"

print("✅ Azure OpenAI client configured")

✅ Azure OpenAI client configured


In [15]:
def generate_schedule_with_llm(new_project_dict, similar_schedules_text):
    """
    Use Azure OpenAI GPT-4o to generate a recommended schedule 
    for the new project based on similar historical schedules.
    """
    
    # Build a detailed system prompt (instructions for the AI)
    system_prompt = """You are an expert construction project scheduler with 20+ years of experience 
in the construction industry. Your job is to create a detailed project schedule 
(Work Breakdown Structure / WBS) for a new construction project.

You will be given:
1. Details of the new project
2. Schedule examples from similar completed projects

Your output must be a structured WBS schedule in the following format:
- Phase/Parent Task Name
  - Sub-task: [Task Name] | Estimated Duration: [X days] | Notes: [brief notes]

Be practical, realistic, and based on the historical examples provided."""

    # Build the user message (the actual question/request)
    user_message = f"""
Please generate a detailed project schedule for the following new project:

=== NEW PROJECT DETAILS ===
- Project Type: {new_project_dict.get('PROJECT_TYPE', 'Not specified')}
- Core Market: {new_project_dict.get('CORE_MARKET', 'Not specified')}
- Market Sector: {new_project_dict.get('MARKET_SECTOR', 'Not specified')}
- Region: {new_project_dict.get('REGION_NAME', 'Not specified')}
- Estimated Budget: ${new_project_dict.get('TOTAL_CONTROL_ESTIMATE', 'Not specified'):,}
- Building Size: {new_project_dict.get('BUILDING_SIZE', 'Not specified')} sq ft

=== HISTORICAL SCHEDULES FROM SIMILAR PROJECTS ===
{similar_schedules_text[:8000]}  

Based on the above historical schedules, generate a recommended project schedule 
for this new project. Include all major phases and sub-tasks.
"""
    
    print("🤖 Calling Azure OpenAI GPT-4o...")
    
    # Make the API call
    response = client.chat.completions.create(
        model = DEPLOYMENT_NAME,
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message}
        ],
        max_tokens = 2000,
        temperature = 0.3  # lower = more focused/consistent output
    )
    
    # Extract the generated text from response
    generated_schedule = response.choices[0].message.content
    
    print("✅ Schedule generated by GPT-4o!")
    return generated_schedule

print("✅ LLM function ready")

✅ LLM function ready


In [17]:

# Step 1: Define a NEW project (this is your input)
new_project = {
    'PROJECT_TYPE':             'Hospitals',
    'CORE_MARKET':              'Clinical Infrastructure',
    'MARKET_SECTOR':            'CI-Healthcare Provider',
    'REGION_NAME':              'Northeast',
    'BUSINESS_UNIT':            'Boston',
    'STATE':                    'MA',
    'TOTAL_CONTROL_ESTIMATE':   1200000,
    'BUILDING_SIZE':            2000
}


print(" NEW PROJECT INPUT")

for k, v in new_project.items():
    print(f"  {k}: {v}")

# Step 2: Find similar historical projects

print(" Finding Similar Projects (ML)")

similar_ids = find_similar_projects(new_project, top_n=3)

#  Retrieve their schedules
print("📋 STEP 3: Retrieving Schedules")
schedule_context = retrieve_schedules(similar_ids)

# Generate new schedule using LLM
print("🤖 STEP 4: Generating Schedule with GPT-4o")
final_schedule = generate_schedule_with_llm(new_project, schedule_context)

# Display final output
print("✅ FINAL RECOMMENDED SCHEDULE")
print(final_schedule)

 NEW PROJECT INPUT
  PROJECT_TYPE: Hospitals
  CORE_MARKET: Clinical Infrastructure
  MARKET_SECTOR: CI-Healthcare Provider
  REGION_NAME: Northeast
  BUSINESS_UNIT: Boston
  STATE: MA
  TOTAL_CONTROL_ESTIMATE: 1200000
  BUILDING_SIZE: 2000
 Finding Similar Projects (ML)
   Feature count being passed to scaler: 7
   Scaler expects: 7 features

🔍 Top 3 similar projects found:
  → PROJECT_0524 | Similarity Score: 1.0000
  → PROJECT_0529 | Similarity Score: 1.0000
  → PROJECT_0519 | Similarity Score: 0.9999
📋 STEP 3: Retrieving Schedules
✅ Retrieved schedules for 3 projects
   Total characters in schedule text: 7390
🤖 STEP 4: Generating Schedule with GPT-4o
🤖 Calling Azure OpenAI GPT-4o...
✅ Schedule generated by GPT-4o!
✅ FINAL RECOMMENDED SCHEDULE
Below is the recommended Work Breakdown Structure (WBS) schedule for the new hospital construction project. The schedule is based on historical examples provided and tailored to the scope and size of the new project:

---

### **Work Breakdown